# FurFriends: train the AI assistant

This notebook trains the FurFriends assistant on the team's data with a free GPU, tests how accurate it is,
and gives you the trained model to download.

**How to run it**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Wait about 20–30 minutes. When it finishes, your browser downloads `furfriends-v1.zip`. Keep it: it's your trained model and the grading sheets.

To change anything (version name, number of epochs, using your own Q&A sheet), edit the **Settings** cell below before you run.

## 1. Settings

In [ ]:
VERSION = "v1"                              # name of this model; use v2, v3 … after each retrain
BASE = "Qwen/Qwen2.5-1.5B-Instruct"         # base model from Hugging Face (Apache 2.0 license)
EPOCHS = 3                                  # passes over the training data
MAX_STEPS = -1                              # -1 = full training; e.g. 5 for a quick trial run
TEST_BASE_MODEL = True                      # also test the untrained model, to show what training added
UPLOAD_QA = False                           # True = upload your own qa.csv (from Google Sheets) instead of the one on GitHub
SAVE_TO_DRIVE = False                       # True = also save the results in your Google Drive

REPO = "https://github.com/DevShinnn/furfriends.git"
BRANCH = "main"

## 2. Check the GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Go to Runtime -> Change runtime type -> T4 GPU -> Save, then Runtime -> Run all again.")
print("GPU:", torch.cuda.get_device_name(0))

## 3. Get the FurFriends code and install the training tools

In [ ]:
import os
os.chdir("/content")
if os.path.exists("furfriends"):
    !git -C furfriends pull -q
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO} furfriends
os.chdir("/content/furfriends/ai-server")
!pip install -q -r requirements-train.txt -r requirements.txt
!git log --oneline -1

## 4. Prepare the data
This uses `data/qa.csv` (your team's Q&As) plus the open data in `data/open/` (722 Pets Stack Exchange Q&As).
With `UPLOAD_QA = True`, you pick the CSV you downloaded from your Google Sheet.

In [ ]:
if UPLOAD_QA:
    from google.colab import files
    print("Choose the qa.csv file you downloaded from Google Sheets (File -> Download -> .csv)")
    up = files.upload()
    name = next(iter(up))
    os.replace(name, "data/qa.csv")
    print("Using your", name)
!python prepare_data.py

## 5. Train
On a T4 this takes about 10–20 minutes for 3 epochs. `loss` should go down. `eval_loss` is measured on the test questions,
which the model never trains on.

In [ ]:
!python train_lora.py --data data/train.jsonl --eval data/test.jsonl --base {BASE} --out adapters/{VERSION} --epochs {EPOCHS} --max-steps {MAX_STEPS}

## 6. Accuracy test
Asks the trained model every test question and writes a grading sheet to `results/`. With `TEST_BASE_MODEL = True`, it
also asks the untrained base model, so you can compare them. Grading is done by people afterwards: see
`docs/ai-evaluation-guide.md`.

In [ ]:
!python evaluate.py run --test data/test.jsonl --train data/train.jsonl --base {BASE} --adapter adapters/{VERSION} --version {VERSION}
if TEST_BASE_MODEL:
    !python evaluate.py run --test data/test.jsonl --train data/train.jsonl --base {BASE} --version base

## 7. Try it

In [ ]:
import sys
sys.path.insert(0, os.getcwd())             # the ai-server folder, so Python finds app.py
import app
from prompting import build_messages
app.load_model(BASE, f"adapters/{VERSION}")

def ask(question, pet=None):
    print(app.generate(build_messages([{"role": "user", "content": question}], pet)))

ask("My puppy has diarrhea since this morning. What should I do?")

In [ ]:
# Ask your own question: change the text and run this cell again.
ask("How often should I brush my cat's teeth?")

## 8. Download the results

In [ ]:
import shutil
os.makedirs("results", exist_ok=True)
pack = f"/content/furfriends-{VERSION}"
os.makedirs(pack, exist_ok=True)
shutil.copytree(f"adapters/{VERSION}", f"{pack}/adapters/{VERSION}", dirs_exist_ok=True)
shutil.copytree("results", f"{pack}/results", dirs_exist_ok=True)
for f in ("data/train.jsonl", "data/test.jsonl"):
    shutil.copy(f, pack)
zip_path = shutil.make_archive(pack, "zip", pack)
print("Made", zip_path)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(zip_path, "/content/drive/MyDrive/")
    print("Saved to Google Drive: My Drive /", os.path.basename(zip_path))

from google.colab import files
files.download(zip_path)

## What's in the zip, and what to do next
- `adapters/v1/` is your trained model, a LoRA adapter of a few tens of MB. Copy it into the `ai-server/adapters/` folder on your computer, then start the server with `ADAPTER_PATH=adapters/v1` and `MODEL_VERSION=v1` (README step 3).
- `results/answers-v1.csv` and `results/answers-base.csv` are the grading sheets. Make one copy per grader, grade them, then run `python evaluate.py score ...` (see `docs/ai-evaluation-guide.md`).
- `train.jsonl` and `test.jsonl` are the exact data this version was trained and tested on. Keep them with the thesis.

To retrain later with more data, update `qa.csv` (or the open data), set `VERSION = "v2"`, and run all again.